In [116]:
import numpy as np

### Data Preprocessing

The dataset is loaded and preprocessed by:
- Normalizing pixel values
- Converting labels into binary classes (0 vs not 0)
- Splitting data into training, validation, and test sets
- Applying feature extraction methods such as flattening, PCA, or HOG

This step prepares the data for efficient model training.

In [117]:
from preprocessing import preprocess
X_train, y_train, X_val, y_val, X_test, y_test, weights = preprocess(feature_method="pca", n_pca=100)

Loading MNIST dataset...
Split completed: Train=54000, Val=6000, Test=10000


### Sigmoid Activation Function

The sigmoid function is used to map the linear output of the model to a probability value between 0 and 1.
This allows the model to interpret outputs as probabilities for binary classification.

In [ ]:
def sigmoid(z):
    return 1.0/(1+np.exp(-z))

### Weighted Cross-Entropy Loss

This function computes the binary cross-entropy loss with class weights to handle data imbalance. Since the dataset contains fewer samples of digit "0", higher importance is assigned to this class.

The loss penalizes incorrect predictions more heavily for the minority class, improving the model’s ability to detect it.

In [119]:

def compute_loss(y, y_hat, weights):
    epsilon = 1e-15
    y_hat = np.clip(y_hat, epsilon, 1 - epsilon)
    
    loss = - (
        weights[1] * y * np.log(y_hat) +
        weights[0] * (1 - y) * np.log(1 - y_hat)
    )
    
    return np.mean(loss)

### Gradient Computation

This function computes the gradients of the loss function with respect to the model parameters (weights and bias).

The gradients indicate how much each parameter should change to minimize the loss. Class weights are applied to ensure that errors on the minority class have a greater impact on the updates.

In [120]:
def compute_gradients(X, y, y_hat, weights):
    m = len(y)
    # apply weights per sample
    sample_weights = np.where(y == 1, weights[1], weights[0])
    error = (y_hat - y) * sample_weights
    dw = (1/m) * (X.T @ error)
    db = (1/m) * np.sum(error)
    
    return dw, db

### Model Training (Fit Function)

This function trains the logistic regression model using gradient descent.

At each iteration:
- The model computes predictions using the sigmoid function
- The loss is calculated
- Gradients are computed
- Parameters are updated

The process is repeated for a fixed number of iterations to optimize the model.

In [122]:
def fit(X, y, weights, iterations=300, lr=0.05):
    
    n_features = X.shape[1]
    w = np.zeros(n_features)
    b = 0
    
    for i in range(iterations):
        z = X @ w + b
        y_hat = sigmoid(z)
        
        # Loss
        loss = compute_loss(y, y_hat, weights)
        
        # compute Gradients ( dL/dw , dL/db)
        dw, db = compute_gradients(X, y, y_hat, weights)
        
        # Update parameters
        w -= lr * dw
        b -= lr * db
        
        if i % 100 == 0:
            print(f"Iteration {i}, Loss: {loss:.4f}")
    
    return w, b

### Prediction Function

This function generates predictions for input data using the trained model.

It first computes the probability using the sigmoid function, then applies a threshold to convert probabilities into binary class labels (0 or 1).

In [123]:
def predict(X,W,b):
    z = X @ W + b
    y_hat=sigmoid(z)
    return (y_hat >= 0.3).astype(int)


In [124]:
# Train model
w, b = fit(X_train, y_train, weights, iterations=1200, lr=1)
# Validation performance
y_val_pred = predict(X_val, w, b)
# Test performance
y_test_pred = predict(X_test, w, b)

Iteration 0, Loss: 0.6931
Iteration 100, Loss: 0.0735
Iteration 200, Loss: 0.0633
Iteration 300, Loss: 0.0595
Iteration 400, Loss: 0.0576
Iteration 500, Loss: 0.0564
Iteration 600, Loss: 0.0556
Iteration 700, Loss: 0.0551
Iteration 800, Loss: 0.0547
Iteration 900, Loss: 0.0544
Iteration 1000, Loss: 0.0542
Iteration 1100, Loss: 0.0540


In [125]:
def evaluate(X, y, w, b, dataset_name="Validation"):
    y_pred = predict(X, w, b)
    y_true = y

    # Positive class = 0 (digit zero)
    TP = np.sum((y_pred == 0) & (y_true == 0))
    TN = np.sum((y_pred == 1) & (y_true == 1))
    FP = np.sum((y_pred == 0) & (y_true == 1))
    FN = np.sum((y_pred == 1) & (y_true == 0))

    accuracy  = (TP + TN) / (TP + TN + FP + FN)
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
    recall    = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    f1        = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0

    print(f"--- {dataset_name} Results ---")
    print(f"Accuracy  : {accuracy:.4f}")
    print(f"Precision : {precision:.4f}")
    print(f"Recall    : {recall:.4f}")
    print(f"F1-Score  : {f1:.4f}")

    print("Confusion Matrix:")
    print(f"                 Predicted 0   Predicted 1")
    print(f"  Actual 0   :   {TP:<12}  {FN}")
    print(f"  Actual 1   :   {FP:<12}  {TN}")

### Model Evaluation

This function evaluates the model performance using:
- Accuracy
- Precision
- Recall
- F1-score
- Confusion Matrix

The evaluation focuses on the minority class (digit 0), which is treated as the positive class to better assess the model’s ability to detect it.

In [126]:
# Evaluate
evaluate(X_val, y_val, w, b, "Validation")
evaluate(X_test, y_test, w, b, "Test")

--- Validation Results ---
Accuracy  : 0.9877
Precision : 0.9104
Recall    : 0.9693
F1-Score  : 0.9389
Confusion Matrix:
                 Predicted 0   Predicted 1
  Actual 0   :   569           18
  Actual 1   :   56            5357
--- Test Results ---
Accuracy  : 0.9876
Precision : 0.9038
Recall    : 0.9776
F1-Score  : 0.9392
Confusion Matrix:
                 Predicted 0   Predicted 1
  Actual 0   :   958           22
  Actual 1   :   102           8918
